In [1]:
import polars as pl
import numpy as np
from scipy import stats
from pathlib import Path

In [2]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/17_Pairwise Ranking HepG2 GSE76344/')
DATASET_PATH = WORKING_PATH / 'dataset'
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined'

In [3]:
# One sample test
def one_sample_test_against_chance(accuracies, p_null=0.5):
    accuracies = np.asarray(accuracies, dtype=float)
    n = len(accuracies)
    mean = accuracies.mean()
    sd = accuracies.std(ddof=1)
    se = sd / np.sqrt(n)
 
    # one-sample t-test -- preferred for small n (e.g. n=5 runs)
    t_stat, t_p = stats.ttest_1samp(accuracies, popmean=p_null)
 
    # z-test shown for reference; not appropriate for n this small
    z_stat = (mean - p_null) / se
    z_p = 2 * (1 - stats.norm.cdf(abs(z_stat)))
 
    return {
        "n_runs": n,
        "mean_accuracy": mean,
        "sd": sd,
        "t_stat": t_stat,
        "t_p_value": t_p,
        "z_stat": z_stat,
        "z_p_value": z_p,
    }

In [4]:
# Pretty report
def report(name, accuracies):
    r = one_sample_test_against_chance(accuracies)
    print(f"{name}:")
    print(f"  mean = {r['mean_accuracy']*100:.2f}% +/- {r['sd']*100:.2f}% "
          f"(n={r['n_runs']} runs)")
    print(f"  t-test:  t({r['n_runs']-1}) = {r['t_stat']:.3f}, "
          f"p = {r['t_p_value']:.4g}")
    print(f"  z-test:  z = {r['z_stat']:.3f}, p = {r['z_p_value']:.4g}")
    print()

In [5]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [6]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",42,67.5,0.77,67.2,0.7593,0.6434,0.7231,0.6809,0.903
"""LogisticRegression""",1011,"""H3K4me3""",null,69.5,0.7562,70.5,0.7447,0.7234,0.6322,0.6748,0.752
"""RandomForest""",1011,"""H3K4me3""",null,70.3,0.7907,69.5,0.775,0.7039,0.6384,0.6696,0.741
"""SVM_Linear""",1011,"""H3K4me3""",null,69.3,0.756,70.6,0.745,0.7251,0.6322,0.6755,0.751
"""DirectRanker""",123,"""H3K4me3""",46,66.3,0.74,68.3,0.7495,0.6508,0.7386,0.6919,0.902
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,66.7,0.7301,64.7,0.6934,0.6401,0.5903,0.6142,0.702
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",100,68.1,0.75,70.8,0.7875,0.681,0.7296,0.7045,0.976
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,65.5,0.7029,68.3,0.7319,0.6802,0.6331,0.6558,0.701


# Chek for the H3K9me3 results

In [7]:
# Check for the H3K9me3
H3K9me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K9me3"))
H3K9me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K9me3""",100,56.7,0.59,56.7,0.6106,0.5399,0.7128,0.6144,0.749
"""DirectRanker""",123,"""H3K9me3""",100,58.1,0.61,56.8,0.6128,0.5425,0.6618,0.5963,0.78
"""DirectRanker""",42,"""H3K9me3""",100,57.0,0.62,56.5,0.6145,0.5244,0.6925,0.5968,0.809
"""DirectRanker""",456,"""H3K9me3""",100,57.1,0.61,55.0,0.5836,0.5212,0.6702,0.5864,0.78
"""DirectRanker""",789,"""H3K9me3""",97,52.6,0.56,56.5,0.6184,0.5342,0.6876,0.6013,0.779


In [8]:
# Select the accuracy
test_accuracies = H3K9me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.567, 0.568, 0.565, 0.55 , 0.565])

In [9]:
report("H3K9me3", test_accuracies)

H3K9me3:
  mean = 56.30% +/- 0.74% (n=5 runs)
  t-test:  t(4) = 19.082, p = 4.444e-05
  z-test:  z = 19.082, p = 0



# Check for H3K27me3

In [10]:
# Check for the H3K27me3
H3K27me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K27me3"))
H3K27me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K27me3""",100,58.4,0.65,59.9,0.6527,0.5601,0.7996,0.6587,0.616
"""DirectRanker""",123,"""H3K27me3""",100,58.5,0.62,62.1,0.6821,0.5752,0.8174,0.6752,0.616
"""DirectRanker""",42,"""H3K27me3""",100,60.0,0.67,62.0,0.6801,0.5619,0.8301,0.6701,0.618
"""DirectRanker""",456,"""H3K27me3""",100,57.6,0.65,57.6,0.6199,0.5363,0.8067,0.6443,0.59
"""DirectRanker""",789,"""H3K27me3""",68,59.7,0.66,57.5,0.6559,0.5355,0.8218,0.6485,0.598


In [11]:
# Select the accuracy
test_accuracies = H3K27me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.599, 0.621, 0.62 , 0.576, 0.575])

In [12]:
report("H3K27me3", test_accuracies)

H3K27me3:
  mean = 59.82% +/- 2.25% (n=5 runs)
  t-test:  t(4) = 9.755, p = 0.0006186
  z-test:  z = 9.755, p = 0

